# 对齐现货区间与持仓量观测

## 目标

给已结束的5分钟现货区间附上当时可得且未过期的持仓量；缺失或过旧时保留空值，不推导资金费率或交易信号。

本文件使用虚构教学数据，不是论文复现或生产数据。

## 准备

使用 Python 3.10+ 内核，按顺序运行全部单元格。计算仅依赖标准库，无需密钥、联网或额外数据文件。可在已有的 Jupyter 环境中打开。

输入已内嵌，与同目录 inputs.json 内容一致；可在下一个单元格中修改 args 试验。时间与单位必须显式保留。

In [ ]:
import json

# Synthetic inputs; no credentials or network access.
bundle = json.loads("{\"version\":1,\"tutorial\":\"crypto-observation-alignment\",\"identity\":\"synthetic\",\"args\":[[{\"openTime\":\"2025-01-06T00:00:00Z\",\"endExclusive\":\"2025-01-06T00:05:00Z\",\"close\":100},{\"openTime\":\"2025-01-06T00:05:00Z\",\"endExclusive\":\"2025-01-06T00:10:00Z\",\"close\":101},{\"openTime\":\"2025-01-06T00:10:00Z\",\"endExclusive\":\"2025-01-06T00:15:00Z\",\"close\":102}],[{\"observedAt\":\"2025-01-06T00:04:00Z\",\"firstSeenAt\":\"2025-01-06T00:06:00Z\",\"value\":10,\"unit\":\"BTC\"},{\"observedAt\":\"2025-01-06T00:09:00Z\",\"firstSeenAt\":\"2025-01-06T00:09:30Z\",\"value\":12,\"unit\":\"BTC\"}],300,\"BTC\"],\"expected\":[{\"openTime\":\"2025-01-06T00:00:00Z\",\"endExclusive\":\"2025-01-06T00:05:00Z\",\"close\":100,\"oiObservedAt\":null,\"ageSeconds\":null,\"openInterest\":null,\"unit\":\"BTC\",\"status\":\"no_available_observation\"},{\"openTime\":\"2025-01-06T00:05:00Z\",\"endExclusive\":\"2025-01-06T00:10:00Z\",\"close\":101,\"oiObservedAt\":\"2025-01-06T00:09:00Z\",\"ageSeconds\":60,\"openInterest\":12,\"unit\":\"BTC\",\"status\":\"aligned\"},{\"openTime\":\"2025-01-06T00:10:00Z\",\"endExclusive\":\"2025-01-06T00:15:00Z\",\"close\":102,\"oiObservedAt\":\"2025-01-06T00:09:00Z\",\"ageSeconds\":360,\"openInterest\":null,\"unit\":\"BTC\",\"status\":\"stale\"}]}")
args = bundle["args"]
expected = bundle["expected"]
print(json.dumps(args, ensure_ascii=False, indent=2))

## 步骤

### 1. 不把同名标的当成同一产品

先固定交易场所、现货对、永续合约、基础资产与计价币，再将单一匹配组传给函数。示例借用BTC单位表达方式，但所有数值虚构。持仓量数量与持仓量名义价值是不同字段，不能因为同属加密数据就用USDT金额代替BTC数量。

### 2. 把时钟映射写清楚

示例采用UTC整秒，现货区间左闭右开，决策时点为区间结束边界。原始毫秒时间需要显式转换并保留原值。持仓量的观察时间描述统计时点，首次观察时间描述本系统何时拿到它；不能仅按统计时间把后来取得的数据放回过去。

### 3. 向后连接并限制年龄

只考虑统计时点与首次观察时点均不晚于区间结束的记录，再选统计时点最新的一条。示例允许边界时刻已经可得的观测，并将最大年龄设为300秒，这是教学约定而非交易所承诺。超过上限标记stale，空值不做向前填充；没有候选则明确保留缺失。

### 4. 检查迟到和单位冲突

第一个区间中的持仓量直到区间结束后才观察到，因此不能使用；第二个区间可连接，第三个区间的观测过旧。重复统计时点、错误单位和非5分钟区间会被拒绝。即使连接成功，也只说明满足示例时间规则，不说明价格和持仓量有因果关系。

### 方法与假设

- 持仓量是存量，不是成交量；不要跨区间求和。
- 资金费率、溢价指数和持仓量不能互相代替。
- 目录合同不证明当前数据可得；网络、权限、历史窗口与首次观察证据仍需核对。

In [ ]:
from datetime import datetime, timezone
import math


def align_spot_and_open_interest(bars, observations, max_age_seconds, expected_unit):
    def parse(value):
        try:
            parsed = datetime.strptime(value, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
            if parsed.strftime("%Y-%m-%dT%H:%M:%SZ") != value:
                raise ValueError()
            return int(parsed.timestamp())
        except (ValueError, TypeError):
            raise ValueError("utc_seconds_required")

    def finite(value):
        return not isinstance(value, bool) and isinstance(value, (int, float)) and math.isfinite(value)

    if not finite(max_age_seconds) or int(max_age_seconds) != max_age_seconds or not 0 <= max_age_seconds <= 86400 or not isinstance(expected_unit, str) or not expected_unit:
        raise ValueError("invalid_alignment_contract")
    oi_times, oi = set(), []
    for row in observations:
        time = parse(row.get("observedAt"))
        available = max(time, parse(row.get("firstSeenAt")))
        if time in oi_times:
            raise ValueError("duplicate_observation")
        oi_times.add(time)
        if not finite(row.get("value")) or row["value"] < 0 or row.get("unit") != expected_unit:
            raise ValueError("invalid_open_interest_unit_or_value")
        oi.append((row, time, available))
    oi.sort(key=lambda item: -item[1])
    bar_times, result = set(), []
    for bar in sorted(bars, key=lambda row: parse(row.get("endExclusive"))):
        start, end = parse(bar.get("openTime")), parse(bar.get("endExclusive"))
        if end - start != 300 or start % 300 != 0 or start in bar_times:
            raise ValueError("invalid_or_duplicate_bar")
        bar_times.add(start)
        if not finite(bar.get("close")) or bar["close"] <= 0:
            raise ValueError("invalid_close")
        candidate = next((item for item in oi if item[1] <= end and item[2] <= end), None)
        age = end - candidate[1] if candidate else None
        status = "no_available_observation" if not candidate else "stale" if age > max_age_seconds else "aligned"
        result.append({**bar, "oiObservedAt": candidate[0]["observedAt"] if candidate else None, "ageSeconds": age, "openInterest": candidate[0]["value"] if status == "aligned" else None, "unit": expected_unit, "status": status})
    return result


### 运行小样本

三个区间依次为no_available_observation、aligned、stale，持仓量分别为null、12、null。

In [ ]:
result = align_spot_and_open_interest(*args)
print(json.dumps(result, ensure_ascii=False, indent=2))

## 检查

将每一行与网页示例的预期输出比较。修改输入后，断言失败可能正是预期结果：先解释差异，不要直接删除验证。

In [ ]:
assert result == expected, "Output differs from the reference synthetic example"
assert bundle["identity"] == "synthetic"
print("通过：结果与网页虚构示例一致。")

## 下一步

真实数据须先通过已认证 GET /v1/catalog 核对权限、字段、schema_major、窗口与来源，再按实际合同映射。这里列出的是候选输入身份，不保证可用或历史完整。不要把 API as_of 当作历史财报版本。真实输入替换后须重新验证；不要沿用这份小样本的通过结论。

- `crypto.spot.binance.btcusdt.5m`
- `crypto.perp.binance.btcusdt.open_interest`

### 参考资料

- [Binance：现货K线时间字段](https://developers.binance.com/docs/binance-spot-api-docs/rest-api/market-data-endpoints)
- [Binance：持仓量统计字段](https://developers.binance.com/docs/derivatives/usds-margined-futures/market-data/rest-api/Open-Interest-Statistics)

[返回教程](https://tradingdatas.com/recipes/crypto-observation-alignment/)